# REDLINE parallel search: Screen 1 of 3

CPU only. Enable Internet, then Save Version > Save & Run All. Each notebook embeds the exact same checksummed source.

This notebook owns partition 1: **675 of the 2,024 default species trios**. Run all three screen notebooks concurrently if your Kaggle account has room. The incumbent is repeated as a control. Each screen eliminates weak teams, mutates survivor moves/items/abilities/EVs/natures/order, and exports six finalists.

The first round uses eight games per candidate (four side-reversed opponent pairs per policy). Ties use remaining HP margin. This costs more than the original four-game screen and still has substantial sampling noise. It does not enumerate all possible sets.

At completion, save the output and add this notebook as an input to the final comparison notebook. Its compact redline-screen-1.zip contains data only. Downloading and re-uploading this ZIP as a dataset also works.

The search pauses at its time/battle budget with a checkpoint. To resume in another session, attach that checkpoint ZIP and set RESUME_ZIP. Keep profile, seed and shard identity unchanged. Only completed jobs are cached; an interrupted job may need rerunning. No competition entry is submitted.


In [ ]:
MODE = 'screen'
SHARD_INDEX = 0  # zero-based; do not change between resumes
SHARD_COUNT = 3
PROFILE = "full"  # use "smoke" in ALL FOUR notebooks for a setup check
MAX_HOURS = 6
MAX_GAMES = 30000
WORKERS = 2
SEED = 43179
RESUME_ZIP = ""
# Final notebook only: leave empty to detect compact screening ZIPs in /kaggle/input.
SCREENING_ZIPS = []


In [ ]:
import base64, hashlib, io, json, os, pathlib, shutil, subprocess, sys, tarfile, urllib.request, zipfile
WORK = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle').exists() else pathlib.Path.cwd() / 'redline-working'
WORK.mkdir(parents=True, exist_ok=True)
PROJECT = WORK / 'redline-project'
RESULTS = WORK / ('redline-final-results' if MODE == 'finalize' else f'redline-screen-{SHARD_INDEX+1}-results')

def extract_zip(data, target):
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(data) as archive:
        for member in archive.infolist():
            destination = (target / member.filename).resolve()
            if target.resolve() not in destination.parents:
                raise ValueError('Unsafe archive member: ' + member.filename)
            if member.is_dir():
                destination.mkdir(parents=True, exist_ok=True)
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(archive.read(member))

BUNDLE_SHA256 = 'a6ac61f80989e32d6d2319476718c4019a5af66d3c25564a375aa695a41652af'
BUNDLE = '...'

data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256
extract_zip(io.BytesIO(data), PROJECT)
if RESUME_ZIP:
    if RESULTS.exists() and any(RESULTS.iterdir()):
        raise ValueError('Results already exist: clear RESUME_ZIP to continue this session.')
    extract_zip(RESUME_ZIP, RESULTS)
print('Verified embedded project:', BUNDLE_SHA256)


In [ ]:
# Install a pinned official Node runtime only if Node >=20 is unavailable.
import platform
node = shutil.which('node')
major = int(subprocess.check_output([node, '--version'], text=True).strip().lstrip('v').split('.')[0]) if node else 0
if major < 20:
    if platform.system() != 'Linux':
        raise RuntimeError('Install Node.js >=20, then rerun this cell. Kaggle Linux installs automatically.')
    arch = {'x86_64': 'x64', 'aarch64': 'arm64'}[platform.machine()]
    name = f'node-v24.13.0-linux-{arch}'
    base = 'https://nodejs.org/dist/v24.13.0/'
    checksums = urllib.request.urlopen(base+'SHASUMS256.txt', timeout=60).read().decode()
    expected = next(line.split()[0] for line in checksums.splitlines() if line.split()[-1] == name+'.tar.xz')
    binary = urllib.request.urlopen(base+name+'.tar.xz', timeout=120).read()
    assert hashlib.sha256(binary).hexdigest() == expected
    # Extract only files/directories from the official verified archive.
    with tarfile.open(fileobj=io.BytesIO(binary), mode='r:xz') as archive:
        for member in archive.getmembers():
            dest = (WORK/member.name).resolve()
            if WORK.resolve() not in dest.parents: raise ValueError('Unsafe Node archive')
            if member.isdir(): dest.mkdir(parents=True, exist_ok=True)
            elif member.isfile():
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(archive.extractfile(member).read())
                dest.chmod(member.mode)
    # npm is a symlink in the archive; recreate its known official target.
    npm = WORK/name/'bin/npm'
    if not npm.exists(): npm.symlink_to('../lib/node_modules/npm/bin/npm-cli.js')
    os.environ['PATH'] = str(WORK/name/'bin') + os.pathsep + os.environ['PATH']
print(subprocess.check_output(['node', '--version'], text=True).strip())
subprocess.run(['npm', 'ci', '--omit=optional', '--no-audit', '--no-fund'], cwd=PROJECT, check=True)


In [ ]:
# Independent processes inside each notebook handle the three opponent policies.
if MODE == 'finalize':
    if not SCREENING_ZIPS:
        SCREENING_ZIPS = sorted(str(p) for p in pathlib.Path('/kaggle/input').rglob('redline-screen-*.zip'))
    if len(SCREENING_ZIPS) != SHARD_COUNT:
        raise ValueError(f'Attach exactly {SHARD_COUNT} completed compact screening ZIPs; found {len(SCREENING_ZIPS)}. Set SCREENING_ZIPS explicitly if needed.')
    print('Final comparison inputs:', SCREENING_ZIPS)
command = [sys.executable, 'tools/team_search.py', '--profile', PROFILE,
           '--out', str(RESULTS), '--hours', str(MAX_HOURS), '--max-games', str(MAX_GAMES),
           '--workers', str(WORKERS), '--seed', str(SEED), '--mode', MODE,
           '--shard-index', str(SHARD_INDEX), '--shard-count', str(SHARD_COUNT)]
if MODE == 'finalize': command += ['--screening'] + SCREENING_ZIPS
try:
    subprocess.run(command, cwd=PROJECT, check=True)
finally:
    if RESULTS.exists():
        checkpoint = WORK / ('redline-final-checkpoint' if MODE == 'finalize' else f'redline-screen-{SHARD_INDEX+1}-checkpoint')
        shutil.make_archive(str(checkpoint), 'zip', RESULTS)
        print('Checkpoint saved:', str(checkpoint)+'.zip')


In [ ]:
from IPython.display import FileLink, display
if MODE == 'screen' and (RESULTS/'screening.json').exists():
    data = json.loads((RESULTS/'screening.json').read_text())
    print('Screen complete:', data['shardIndex']+1, 'of', data['shardCount'])
    print('Default trios covered:', len(data['populationIndices']), 'completed games:', data['completedGames'])
    compact = RESULTS/f'redline-screen-{SHARD_INDEX+1}.zip'
    # Root output copies are easy to find when adding this notebook as an input.
    shutil.copyfile(compact, WORK/compact.name)
    display(FileLink(str(WORK/compact.name)))
elif (RESULTS/'result.json').exists():
    summary = json.loads((RESULTS/'result.json').read_text())
    print(summary['warning'])
    print('Selected team:', ', '.join(p['name'] for p in summary['selected']['team']))
    print(json.dumps(summary['confirmation'], indent=2))
    display(FileLink(str(RESULTS/'best_candidate.zip')))
elif (RESULTS/'status.json').exists():
    print((RESULTS/'status.json').read_text())
display(FileLink(str(checkpoint)+'.zip'))
print('Save completed output for the final notebook, or download the checkpoint to resume.')
